# Walkthrough: the Paradoxo player that learns by reinforcement

This notebook takes the learned player apart piece by piece. Every cell runs;
change the numbers and run it again. The full explanation is in `rl/README.md`,
and every training run has its own report in `rl/results/runs/<run>/REPORT.md`.

The player makes three kinds of decisions, each with its own neural network
(a "head"): the **dice** every Hour, the **market** every time it can shop, and
the **reward** category after every contract point.

1. A real position, frozen
2. Every legal dice move
3. What the network sees
4. The network, and checking its gradients by hand
5. How the trained player scores the dice moves
6. The market: how it scores every card
7. One gradient step, by hand
8. A whole match, decision by decision
9. The training run: curves, milestones, setbacks
10. The benchmark at every table size
11. Things to try

In [ ]:
import os, sys
root = os.path.abspath("..") if os.path.basename(os.getcwd()) == "rl" else os.getcwd()
os.chdir(root); sys.path.insert(0, root)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, json
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 220)

from rl.lab import (capture_position, capture_market, move_table, market_table, state_table,
                    watch_match, load_heads, latest_run, describe)
from rl.actions import legal_moves, legal_allocations
from rl.features import state_features, move_features
from rl.network import MLP, Adam
from rl.agent import softmax

RUN = latest_run()
base, heads = load_heads(os.path.join(RUN, "best.npz"))
print("run:", RUN, "| built on:", base, "| heads:", [h for h in heads if h != "value"])

## 1. A real position, frozen

`capture_position(seed, hour, n_players=...)` plays match number `seed` with the
Aggressive heuristic in the test seat and freezes it at that Hour's dice phase.
You get the traveler, the whole game and the four dice, as copies you can change.

In [ ]:
SEED, HOUR, PLAYERS = 7, 8, 4
traveler, game, dice = capture_position(SEED, HOUR, n_players=PLAYERS)
print("dice:", dice)
print("century", traveler.century, "| energy", traveler.energy, "| gold", traveler.gold,
      "| CP", traveler.contract_points, "| booms", traveler.booms)
print("hand:", [(c.name, c.delivery_century) for c in traveler.hand])
print("Merchant on", game.merchant_century, "| overloaded:", traveler.overloaded_functions,
      "| scored XX:", traveler.scored_century_xx, "| scored X:", traveler.scored_century_x)

## 2. Every legal dice move

The rules (§10, §11) turn 4^4 = 256 ways to throw four dice at three functions
into a few distinct legal placements. Each placement that travels is offered in
both directions, with no cap and with caps that stop exactly on a century worth
stopping on: a delivery, the Merchant, the checkpoints X and XX not yet scored,
and the Secret Market on XI.

Try it: change `dice` to `[3, 3, 3, 3]`, or put the traveler on century 24 and
look for a move that stops on XX.

In [ ]:
moves = legal_moves(traveler, game, dice)
print(len(legal_allocations(dice, traveler.overloaded_functions)), "placements,", len(moves), "moves")
for m in moves:
    print(" ", describe(m))

## 3. What the network sees

The position (the same for every option) and the option. For a dice move the
option is an estimate of what the move would do: where it lands, the energy
after, the booms, whether it lands on a checkpoint.

In [ ]:
state_table(traveler, game)

In [ ]:
move_table(traveler, game, dice)[["move", "heuristic's pick", "centuries travelled", "landing century",
                                  "energy after", "booms after", "lands on XX unscored",
                                  "lands on X unscored", "lands on XI"]]

## 4. The network, and checking its gradients by hand

A multilayer perceptron written in numpy: `h = tanh(x W + b)` twice, then a
linear output. The backward pass is written by hand, so it has to be checked:
nudge one weight by a tiny `eps` up and down and compare the change in the
output with what the backward pass predicted.

In [ ]:
rng = np.random.default_rng(0)
net = MLP([5, 8, 1], rng)
x = rng.normal(size=(4, 5))
out, acts = net.forward(x)
grads = net.backward(acts, np.ones_like(out))

eps, (i, j) = 1e-6, (2, 3)
W = net.params[0]; old = W[i, j]
W[i, j] = old + eps; up = net(x).sum()
W[i, j] = old - eps; down = net(x).sum()
W[i, j] = old
print("finite difference:", (up - down) / (2 * eps))
print("backward pass:    ", grads[0][i, j])

## 5. How the trained player scores the dice moves

The dice head scores every move; a softmax turns the scores into probabilities.
In play the player takes the top one. Is it the same as the heuristic's pick?

In [ ]:
move_table(traveler, game, dice, heads)[["move", "score", "probability", "heuristic's pick",
                                         "landing century", "energy after"]]

## 6. The market: how it scores every card

`capture_market(seed)` freezes the first time the player can choose between at
least three things at the market. The market head scores buying each card,
renewing each card, paying off Wanted, or passing. Every card has its own input,
so over hundreds of thousands of matches the network learns what each of the 52
cards is worth in each situation.

In [ ]:
m_traveler, m_game, revealed, renew_cost = capture_market(seed=5)
print("gold", m_traveler.gold, "| century", m_traveler.century, "| renew cost", renew_cost)
market_table(m_traveler, m_game, revealed, renew_cost, heads)

## 7. One gradient step, by hand

This is the whole of REINFORCE for one decision. Suppose the player picked move
`a` and went on to win (R = 1) from a position the value network rated at
V = 0.35. The advantage is R - V = 0.65, better than expected, so move `a`
becomes more likely. The gradient of log pi(a) with respect to the scores is
`onehot(a) - p`, fed backward through the network.

In [ ]:
dice_head = heads["dice"]
s = state_features(traveler, game)
X = np.stack([np.concatenate([s, move_features(traveler, game, m)]) for m in moves])
scores, acts = dice_head.forward(X)
p = softmax(scores[:, 0])
a = int(np.argsort(p)[-2])          # the second favourite, to watch it rise
R, V = 1.0, 0.35
onehot = np.zeros_like(p); onehot[a] = 1
grads = dice_head.backward(acts, ((R - V) * (onehot - p))[:, None])

trial = MLP(dice_head.sizes); trial.set(dice_head.get())
Adam(trial.params, lr=1e-2).step(trial.params, grads, ascent=True)
p_after = softmax(trial(X)[:, 0])
print("move:", describe(moves[a]))
print(f"probability before {p[a]:.3f}  after one step {p_after[a]:.3f}")

## 8. A whole match, decision by decision

The trained player sits in the test seat. Every dice and market decision is
logged next to what the Aggressive heuristic would have done in the same spot.

In [ ]:
log, winner, reason, final = watch_match(seed=3, heads=heads, n_players=4)
print("winner:", winner, "| ended by:", reason, "| final CP:", final)
print(log.groupby("decision")["same"].mean().rename("agrees with the heuristic"))
log

In [ ]:
log[~log['same']][['hour', 'decision', 'situation', 'century', 'energy', 'gold', 'learned', 'heuristic']]

## 9. The training run: curves, milestones, setbacks

Every run keeps its whole history in its folder. `evals.jsonl` has the greedy
player measured at every table size on matches it never trained on;
`events.jsonl` has the milestones, setbacks, recoveries and resumes.
The **mean edge** is the win rate times the number of players, averaged over
table sizes: 1.0 is a fair share.

In [ ]:
read = lambda f: [json.loads(l) for l in open(os.path.join(RUN, f)) if l.strip()]
its, evals, events = pd.DataFrame(read("iterations.jsonl")), pd.DataFrame(read("evals.jsonl")), pd.DataFrame(read("events.jsonl"))
refs = json.load(open("rl/results/reference_edges.json"))
cfg = json.load(open(os.path.join(RUN, "config.json")))
ref = {h: r["mean_edge"] for h, r in refs[f"{cfg['eval_games']}_{cfg['eval_seed']}"].items()}

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(evals.iteration, evals.mean_edge, "o-", ms=3, label="learned player")
for h, e in ref.items():
    ax[0].axhline(e, ls="--", lw=1, label=f"{h} {e:.2f}")
ax[0].axhline(1, c="k", lw=.5); ax[0].set_xlabel("iteration"); ax[0].set_ylabel("mean edge"); ax[0].legend(fontsize=8)
for n in sorted(evals.by_size.iloc[0], key=int):
    ax[1].plot(evals.iteration, [b[n]["win_rate"] for b in evals.by_size], "o-", ms=2, label=f"{n} players")
ax[1].set_xlabel("iteration"); ax[1].set_ylabel("win rate"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
events[events.kind.isin(["milestone", "setback", "recovery", "resume", "phase"])][["time", "kind", "text"]]

## 10. The benchmark at every table size

The same fixed tables for every player, 2 to 6 players, the subject rotating
through every seat, the same seeds for everyone. A small run here; the full one
is `python -m rl.evaluate --weights <run>/best.npz`.

In [ ]:
from rl.evaluate import benchmark, load_weights
spec = {h: v for h, v in load_weights(os.path.join(RUN, "best.npz"))[1].items() if h != "value"}
rows = []
for n in (2, 3, 4, 5, 6):
    for label, kind, payload in (("Aggressive", "heuristic", "Aggressive"), ("Learned", "learned", (base, spec))):
        r = benchmark(kind, payload, 400, seed=30_000_000, workers=4, n_players=n)
        rows.append({"players": n, "player": label, "win rate": r["win_rate"], "95% CI": r["ci95"], "edge": r["edge"]})
pd.DataFrame(rows).pivot(index="players", columns="player", values=["win rate", "edge"])

## 11. Things to try

- **Section 2**: set `traveler.century = 24` and find the move that stops on XX.
- **Section 6**: change the gold (`m_traveler.gold = 2`) and see which purchases
  disappear and how the scores move.
- **Section 7**: set `R = 0` (a loss). The same move becomes less likely. Why?
- **Section 8**: `n_players=2` and `n_players=6`. Does the player change style?
- **Training**: `python -m rl.train --name test --iterations 50 --entropy 0.05`
  explores more; compare its curves with the main run.